In [ ]:
#!pip install -q lxml
#!pip install unidecode
import pandas as pd
import numpy as np
#import geopandas as gpd
import urllib#pour récupérer les données
import bs4#pour rendre lisibles les données
import lxml
import re
import time
from unidecode import unidecode
import urllib

from urllib import request

In [ ]:
def Scrap (url):
    req = urllib.request.Request(url)
    html = urllib.request.urlopen(req).read()
    page = bs4.BeautifulSoup(html, "lxml")
    return page

In [ ]:
def ToDf (table) :
    Listgrossiere=pd.read_html(str(table))#on transforme le tableau en liste de dataframes
    listeDf=Listgrossiere[0]#on récupère le dataframe
    return listeDf

In [ ]:
page=Scrap('https://fr.wikipedia.org/wiki/Liste_des_noms_fran%C3%A7ais_des_villes_europ%C3%A9ennes')

In [ ]:
tables = page.findAll('table')
table = tables[0]
liste=ToDf(table)
listelien=[]
#for i in range(32):
#    page=Scrap('http://www2.assemblee-nationale.fr/sycomore/resultats/(offset)/'+str((i+1)*500)+'/(query)/IiBTRUxFQ1QgbV9jb2RlX2RlcHV0ZSwgaWRfYWN0ZXVyX3RyaWJ1biwgbGVnX21heF90cmlidW4sIG5vbSwgbm9tX2FmZmljaGUsIHByZW5vbSwgZGF0ZV9uYWlzLCBkYXRlX2RlY2VzIEZST00gZGVwdXRlIFdIRVJFIDE9MSAgQU5EIG1fY29kZV9kZXB1dGUgSU4gKFNFTEVDVCBtX2NvZGVfZGVwdXRlIEZST00gbWFuZGF0LCBkZXBhcnRlbWVudCAgV0hFUkUgbWFuZGF0Lm1fbnVtX2RlcCA9IGRlcGFydGVtZW50Lm1fbnVtX2RlcCBBTkQgbWFuZGF0Lm1fdHlwZV9tYW5kYXQgPSAxICkgT1JERVIgQlkgZGF0ZV9uYWlzIERFU0Mi')
table = page.find('table')#on récupère le seul tableau de la page qui liste des députés
listeDf=ToDf(table)
liste=pd.concat([liste,listeDf])#on récupère le dataframe et on l'ajoute à la liste  
for ele in table.findAll('a'):
    listelien.append(ele.get('href'))
liste=liste.reset_index()#on recrée un index afin de ne pas s'arrêter à 500 puis recommencer comme dans les pages web

In [ ]:
page_dep=Scrap('https://fr.wikipedia.org/wiki/Liste_des_d%C3%A9partements_fran%C3%A7ais')
tables = page_dep.findAll('table')
table = tables[1]
liste_dep=ToDf(table)
liste_dep = liste_dep.droplevel(level = [0, 1], axis = 1)
listelien=[]
#for i in range(32):
#    page=Scrap('http://www2.assemblee-nationale.fr/sycomore/resultats/(offset)/'+str((i+1)*500)+'/(query)/IiBTRUxFQ1QgbV9jb2RlX2RlcHV0ZSwgaWRfYWN0ZXVyX3RyaWJ1biwgbGVnX21heF90cmlidW4sIG5vbSwgbm9tX2FmZmljaGUsIHByZW5vbSwgZGF0ZV9uYWlzLCBkYXRlX2RlY2VzIEZST00gZGVwdXRlIFdIRVJFIDE9MSAgQU5EIG1fY29kZV9kZXB1dGUgSU4gKFNFTEVDVCBtX2NvZGVfZGVwdXRlIEZST00gbWFuZGF0LCBkZXBhcnRlbWVudCAgV0hFUkUgbWFuZGF0Lm1fbnVtX2RlcCA9IGRlcGFydGVtZW50Lm1fbnVtX2RlcCBBTkQgbWFuZGF0Lm1fdHlwZV9tYW5kYXQgPSAxICkgT1JERVIgQlkgZGF0ZV9uYWlzIERFU0Mi')
table = page_dep.find('table')#on récupère le seul tableau de la page qui liste des députés
listeDf=ToDf(table)
liste_dep=pd.concat([liste_dep,listeDf])#on récupère le dataframe et on l'ajoute à la liste  
for ele in table.findAll('a'):
    listelien.append(ele.get('href'))
liste_dep=liste_dep.reset_index()

In [ ]:
simplif = re.compile("(\w+-?)*")

In [ ]:
def simplifie(x):
    return simplif.match(x.replace('ç', 'c').replace('é', 'e')).group()
def unifie(x):
    return x.split("/")[0]

In [ ]:
tables = page.findAll('table')
DF_tot = pd.DataFrame()
for table in tables:
    new_df = ToDf(table)
    new_df.columns = ["Nom local", "Nom en francais"]
    DF_tot=pd.concat([DF_tot,new_df])

In [ ]:
DF_tot.columns = ["Nom local", "Nom en francais"]
DF_tot["Nom en francais"] = DF_tot["Nom en francais"].apply(simplifie)
DF_tot["Nom local"] = DF_tot["Nom local"].apply(unifie).str.rstrip()
DF_tot

In [ ]:
Df_dep = liste_dep[["Nom", "Chef-lieu"]].dropna()

In [ ]:
for column in Df_dep.columns:
    Df_dep[column] = Df_dep[column].apply(simplifie)

In [ ]:
Df_dep = Df_dep[:99]
Df_dep = Df_dep.rename(columns = {"Nom": "Nom en francais", "Chef-lieu":"Nom local"})
Df_dep

In [ ]:
Noms_speciaux = {"Allemagne":"Berlin", "Espagne":"Madrid", "Italie":"Roma", "Aix":"Aix-en-Provence", 
                 "Suisse": "Geneve", "Catalogne": "Barcelona", "Sambre Et Mesuse": "Namur", "Sambre Et Meuse": "Namur", 
                "Hollande": "Amsterdam", "Mont-Blanc": "Chamonix-Mont-Blanc", "Siegberg": "Siegburg",
                "Pouilly-Sur-Loire":"Pouilly-sur-Loire", "Zürich": "Zuerich", "Zurich": "Zuerich", 
                 "Mayence": "Mainz", "Mayence (All)": "Mainz",
                "Anranjuez": "Aranjuez", "Ancône":"Ancona", 'Ancône (Italie)': 'Ancona', 
                 "Montelegino":"Montenotte A", "Épinal": "Epinal", 'Loire Inferieure':'Nantes','Loire Inferieure': 'Nantes', 
                "Orléans":"Orleans", 'Charité Sur Loire':"La Charite-sur-Loire", "Bourg-De-Peage":"Bourg-de-Peage",
                "Beaugency (Loiret)": "Beaugency", "Saint Maximin": "Saint-Maximin", 
                'Valenciennes Condé Landrecies Le Quesnoy': 'Valenciennes', 'Chambly (Oise)': 'Chambly',
                '(Rhone-Et-Loire) Lyon':'Lyon', 'Bréda': 'Breda', 'Camp De Tournoux (Saint-Paul-Sur-Ubaye)': 'Saint-Paul-sur-Ubaye',
                'Neukirch (All)':'Neukirch', 'Port-Vendre': 'Port-Vendres', 'La Châtre':'La Chatre',
                 'Petit-Saint-Bernard':'Seez', 'Laibach (Ljubljana)':'Ljubljana', 'Guéret': 'Gueret', 
                 'Brest ':'Brest', 'Saint-Jean-Pied-De-Port':'Saint-Jean-Pied-de-Port',
                 'Lons-Le-Saunier': 'Lons-le-Saunier', 'Danzig':'M. Gdansk', 'Maestricht':'Maastricht', 
                 'Francfort':'Frankfurt am Main', 'Malte': 'Valette', 'Trèves': 'Trier', 'Trèves Mayence':'Trier',
                 'Hohenlinden (Bavière)':'Hohenlinden', 'Iéna':'Jena', 'Cassel (All)':'Cassel', 'Lunéville':'Luneville',
                 'Escaut':'Gent', 'Saint-Chamans': 'Saint-Chamant', 'Brxuelles': 'Bruxelles','Belgique': 'Bruxelles',
                 'Angleterre': 'London', 'Anvers Belgique': 'Antwerpen', "Condé-Sur-L'Escaut": "Conde-sur-l'Escaut",
                 "Condé-Sur-L'Escaut (Nord)": "Conde-sur-l'Escaut",
                'Usa':'Hors_europe', 'Égypte':'Hors_europe','Étranger': 'Hors_europe','Etats-Unis':'Hors_europe',
                 'Suisse, Hollande': 'Geneve', 'Usa, Uk': 'London', 
                 'Londres':'London', ' Paris':'Paris', 'Pparis':'Paris', 'Piemont':'Torino', 'Piémont': 'Torino',
                 'Montlieu': 'Montolieu','Corfou (Grèce)': 'Kerkyras','Frontière Espagne': 'Andorra',
                  'Italie->Paris':'Roma', 'Jemmapes': 'Mons', 'Alsace': 'Strasbourg','Sur Le Mein': 'Frankfurt am Main',
                 'Elbe Et De La Meuse-Inférieure': 'Maastricht',
                 'Roer, Rhin, Moselle, Mont-Tonnerre': 'Strasbourg', 'De Bâle À Landau':'Basel',
                 'Indre-Et-Loire':'Tours', 'Rhin':'Strasbourg', 'Nord, Ardennes, Moselle': 'Dunkerque', 
                 'Loir Et Cher': 'Blois', 'Basses-Pyrénées':'Pau', 'Basses-Pyrennees': 'Pau', 'Ouest': 'Bordeaux',
                 'Ile-Et-Vilaine': 'Rennes', 'Pyrénées Orientales': 'Perpignan', 'Seine': 'Paris',
                 'Pas-De-Calais,Nord Lys Somme': 'Lille', 'Lot-Et-Garonne': 'Agen', 
                 'Eure Manche Calvados Orne': 'Evreux', 'Corse': 'Bastia', 'Languedoc': 'Toulouse', 
                }
Df_special = pd.DataFrame.from_dict(Noms_speciaux, orient = "index").reset_index()
Df_special.columns = ["Nom en francais", "Nom local"]

In [ ]:
Df_special

In [ ]:
#DF_tot2 = 
DF_tot2 = DF_tot.loc[~(DF_tot["Nom en francais"].isin(Noms_speciaux.keys())),:]
DF_tot2 = pd.concat([DF_tot2, Df_special, Df_dep]).reset_index()
# del DF_tot2["index"]

In [ ]:
DF_tot2

In [ ]:
DF_tot2[DF_tot2["Nom en francais"]=="Aix"]

In [ ]:
DF_tot2[DF_tot2["Nom en francais"]=="Pyrénées-Orientales"]

In [ ]:
#senateur = pd.read_csv("C:/Users/sylva/OneDrive/Bureau/senat/Data/test_17_senateurs.csv")
senateur = pd.read_excel("C:/Users/sylva/OneDrive/Bureau/senat/Data/80_senateurs_modified.xlsx")


In [ ]:
years = list(range(1789, 1816))

In [ ]:
def set_place_at_year(personne, year=1800):
    if year < 1789:
        period = "AR"
        nb_period_max = 1
    if 1788 < year < 1799:
        period = "revolution"
        nb_period_max = 10
    if 1798 < year < 1805:
        period = "consulat"
        nb_period_max = 5
    if 1804 < year < 1816:
        period = "empire"
        nb_period_max = 7
    for num_period in range(1, nb_period_max):
        date_num = personne["date "+period+" "+str(num_period)]
        if date_num == str(date_num):
            if int(year) in [int(date) for date in date_num.replace("/", "-").split("-")]:
                date_num = year
            else:
                date_num = 0
        if pd.notna(date_num) and int(date_num)==int(year):
            lieu = personne["lieu "+period+" "+str(num_period)]
            if (type(lieu)==str) & (lieu != "paris"):
                return personne["lieu "+period+" "+str(num_period)].title().rstrip()
    return "Paris"

In [ ]:
for year in years:
    senateur[year] = senateur.apply(set_place_at_year, axis = 1, args = [year])

In [ ]:
senateur[1792][senateur[1792].str.contains('Brest')].values

In [ ]:
senateur["position_sociale"]=senateur["1. place hiérarchie sociale famille"]

In [ ]:
def to_nom_unique(x, year):
    if x["Nom local"] == str(x["Nom local"]):
        return x["Nom local"].rstrip()
    else:
        return x[year]

In [ ]:
DF_tot2["Nom en francais"] = DF_tot2["Nom en francais"].str.rstrip()

In [ ]:
years
for year in years:
    senateur = pd.merge(senateur, DF_tot2, left_on = year, right_on = "Nom en francais", how = 'left')
    senateur["nom_local"+str(year)] = senateur.apply(to_nom_unique, axis = 1, args = [year])
    del senateur["Nom local"]
    del senateur["Nom en francais"]
    del senateur["index"]

In [ ]:
DF_tot2[DF_tot2["Nom local"]=="Zuerich"]

In [ ]:
senateur.columns

In [ ]:
def to_date_naiss(x):
    return float("17"+str(x)[-2:])
def to_date_nomin(x):
    if float(str(x)[-2:])>50:
      return float("17"+str(x)[-2:])
    else:
      return float("18"+str(x)[-2:])

In [ ]:
senateur["annee naiss"] = senateur["date naiss"].apply(to_date_naiss)
senateur["annee nomin"] = senateur["date nomin"].apply(to_date_nomin)

In [ ]:
senateur_for_map = senateur[["nom_local"+str(year) for year in years]+["position_sociale","annee nomin", "nom", "annee naiss"]]

In [ ]:
senateur_for_map.to_csv("C:/Users/sylva/OneDrive/Bureau/senat/Data/senateur_for_map.csv")

In [ ]:
for col in senateur_for_map:
    if "Zürich" in senateur_for_map[col].to_list():
        print(col)

In [ ]:
senateur